# 00 Device Physics and Trace

Every figure below is constructed in its owning cell. The numerical figures
read the Au measured fixtures and simulate the declared surrogate live; the device-mechanism
schematic is authored directly in Matplotlib. Figures always appear inline.
File export is optional and disabled by default.


## Configuration


In [ ]:
from pathlib import Path
import hashlib, json, os

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

RUN_PROFILE = os.getenv("MRL_RUN_PROFILE", "reduced")  # reduced | publication | smoke
DEVICE = os.getenv("MRL_DEVICE", "auto")
SELECTED_DEVICE = "cpu"  # these measured-data fits are NumPy/SciPy workloads
WORKERS = os.getenv("MRL_WORKERS", "auto")
SAVE_FIGURES = os.getenv("MRL_SAVE_FIGURES", "0") == "1"
OUTPUT_DIR = Path(os.getenv("MRL_OUTPUT_DIR", "generated_figures"))
OVERWRITE = os.getenv("MRL_OVERWRITE", "0") == "1"
RUN_EXTERNAL_DATA = os.getenv("MRL_RUN_EXTERNAL_DATA", "0") == "1"
ALLOW_DATA_DOWNLOADS = os.getenv("MRL_ALLOW_DATA_DOWNLOADS", "0") == "1"

if RUN_PROFILE not in {"reduced", "publication", "smoke"}:
    raise ValueError(f"unknown RUN_PROFILE={RUN_PROFILE!r}")

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "experiments" else HERE
DEVICE_DATA = ROOT / "data" / "device_model"
OWNED = (
    "fig_measured_transient.png", "fig_trace_shapes.png", "fig_dispersive.png",
    "fig_timescales.png",
)
FIGURE_REPORT = []
plt.ioff()

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def finish(fig, filename, provenance_class, *, data_paths=(), seeds=None, details=None, method_provenance=None):
    """Display once, optionally save, close, and record figure provenance."""
    saved = None
    if SAVE_FIGURES:
        out = OUTPUT_DIR.expanduser().resolve()
        if "manuscript" in str(out).lower() and not OVERWRITE:
            raise FileExistsError("writing into a manuscript directory requires OVERWRITE=True")
        out.mkdir(parents=True, exist_ok=True)
        path = out / filename
        if path.exists() and not OVERWRITE:
            raise FileExistsError(path)
        fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
        saved = str(path)

    if filename not in OWNED:
        raise KeyError(f"unregistered notebook figure: {filename}")
    paths = [Path(path) for path in data_paths]
    if method_provenance is None:
        method_provenance = {
            "status": "empirical_fit" if paths else "proposed",
            "established_basis": ["KWW fitting"] if paths else [],
            "repository_adaptation": "notebook-owned analysis or visualization",
            "claim_limit": "descriptive within the measured files or stated simulation settings",
        }
    record = dict(
        filename=filename,
        runtime_profile=RUN_PROFILE,
        runtime_device=SELECTED_DEVICE,
        seeds=seeds,
        data_files=[str(path.relative_to(ROOT)) for path in paths],
        runtime_data_hashes={str(path.relative_to(ROOT)): sha256(path) for path in paths},
        provenance_class=provenance_class,
        saved_path=saved,
        details=details or {},
        method_provenance=method_provenance,
    )
    FIGURE_REPORT.append(record)
    display(fig)
    plt.close(fig)
    return record

print({"profile": RUN_PROFILE, "requested_device": DEVICE, "selected_device": SELECTED_DEVICE, "workers": WORKERS,
       "saving": SAVE_FIGURES, "external": RUN_EXTERNAL_DATA,
       "downloads": ALLOW_DATA_DOWNLOADS})


## Measured Device Transient

Real gold-contact current transients (three repeats per bias) are averaged
and plotted against the global KWW law. The lines are not per-trace refits.


In [ ]:
"""Measured SiOx current transients with the global compressed-exponential fit."""
GOLD_EXPORT = DEVICE_DATA / "gold_export"
KWW_JSON = DEVICE_DATA / "kww_final.json"
VS = [0.8, 0.9, 1.1, 1.2, 1.4, 1.5]

fitted = json.loads(KWW_JSON.read_text(encoding="utf-8"))
laws, rows = fitted["laws"], fitted["rows"]
beta, tr0, cr = laws["beta"], laws["tr0"], laws["cr"]
td0, cd = laws["td0"], laws["cd"]

def model(t, V):
    """Global-law KWW model at bias |V|; A and C come from the per-V row."""
    row = rows[f"{V:.1f}"]
    tau_r = tr0 * np.exp(-cr * V)
    tau_d = td0 * np.exp(-cd * V)
    return row["A"] * (1 - np.exp(-(t / tau_r) ** beta)) * np.exp(-t / tau_d) + row["C"]

def traces_for(V):
    traces = []
    for path in sorted(GOLD_EXPORT.glob(f"trace_Vm{V:.2f}_tr*.csv")):
        values = np.genfromtxt(path, delimiter=",", names=True)
        t = np.asarray(values["time"], float)
        current = np.abs(np.asarray(values["current"], float))
        keep = np.isfinite(t) & np.isfinite(current) & (current > 0)
        traces.append((t[keep], current[keep]))
    if len(traces) != 3:
        raise ValueError(f"expected three measured repeats at {V:.1f} V, found {len(traces)}")
    return traces

fig, ax = plt.subplots(figsize=(6.6, 4.3))
cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(VS)))
for colour, V in zip(cmap, VS):
    traces = traces_for(V)
    tmax = min(t[-1] for t, _ in traces)
    tg = np.linspace(0, tmax, 300)
    mean_current = np.mean([np.interp(tg, t, current) for t, current in traces], axis=0)
    stride = max(1, len(tg) // 90)
    ax.plot(tg[::stride], mean_current[::stride] * 1e9, ".", color=colour, ms=2.6, alpha=0.4)
    ax.plot(tg, model(tg, V) * 1e9, "-", color=colour, lw=1.8, label=f"{V:.1f} V")

ax.set_xlabel("time after bias step (s)")
ax.set_ylabel(r"$|I|$ (nA)")
ax.set_xlim(0, 200)
ax.legend(fontsize=8, frameon=False, ncol=2, title="bias", title_fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, which="major", color="0.85", lw=0.5, zorder=0)
ax.set_axisbelow(True)
fig.tight_layout(pad=0.3)
finish(fig, "fig_measured_transient.png", "live-exact",
       data_paths=[KWW_JSON, *sorted(GOLD_EXPORT.glob("trace_Vm*_tr*.csv"))], seeds=3)


## Eligibility Trace Shapes

A `CascadeEligibilityGate` surrogate is integrated at each deliberately
swept retention setting. It approximates the fitted rise shape; it is not an
exact state-space realization or an identified microscopic stage count.


In [ ]:
"""Proposed cascade eligibility surrogate under a retention sweep."""
from mrl_trace import CascadeEligibilityGate, PRIMARY_MODEL_ID, device_model_spec
MODEL_SPECIFICATION = device_model_spec(PRIMARY_MODEL_ID)

PALETTE = [plt.cm.viridis(x) for x in (0.85, 0.5, 0.15)]
dt = 0.02
t = np.arange(0, 60.0, dt)
tau_leaks = [20.0, 5.0, 1.0]

fig, ax = plt.subplots(figsize=(4.2, 3.0))
metrics = {}
print("tau_leak    peak time (s)    1/e width (s)")
for tau_leak, colour in zip(tau_leaks, PALETTE):
    gate = CascadeEligibilityGate(V=0.9, tau_leak=tau_leak, dt=dt)
    eligibility = gate.trace(t, coincidence_at=2.0, coincidence_dur=0.3, normalise=True)
    ax.plot(t, eligibility, color=colour, lw=1.8, label=fr"$\tau_{{\rm leak}}={tau_leak:g}$ s")
    peak_index = int(np.argmax(eligibility))
    below = np.where(eligibility[peak_index:] <= np.exp(-1))[0]
    width = (t[peak_index + below[0]] - t[peak_index]) if len(below) else np.inf
    metrics[tau_leak] = {"peak_time_s": float(t[peak_index]), "one_over_e_width_s": float(width)}
    print(f"  {tau_leak:5.1f}      {t[peak_index]:7.2f}        {width:7.2f}")

ax.set_xlabel("time after coincidence (s)")
ax.set_ylabel(r"eligibility trace $e(t)$ (norm.)")
ax.set_xlim(0, 60); ax.set_ylim(0, 1.05)
ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, which="major", color="0.85", lw=0.5, zorder=0)
ax.set_axisbelow(True)
fig.tight_layout()
finish(
    fig, "fig_trace_shapes.png", "live-exact", details=metrics,
    method_provenance={
        "status": "proposed",
        "established_basis": ["sequential waiting-time cascade"],
        "repository_adaptation": "cascade-shaped eligibility surrogate with swept leakage",
        "claim_limit": "numerical approximation; no microscopic stage-count identification",
    },
)


## Conceptual Dispersive-Relaxation Hypotheses

This authored schematic illustrates candidate transport interpretations and
qualitative curve shapes. It is not a microscopic identification, and the
illustrative fill and discharge curves do not establish a shared mechanism.


In [ ]:
import os
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

INK   = "#2b2b2b"
MO    = "#9aa6b2"
OXIDE = "#eef1f4"
GOLD  = "#e0a93b"
ITO   = "#3aa07a"
ITOr  = "#6b4f2a"
TRAP  = "#3aa07a"
PROT  = "#c0392b"
GREEN = "#3aa07a"
NEU   = "#9aa6b2"

TRAP_PTS = None   # shared trap distribution, set in main()

def _stack(ax, reduced, top_color, top_label):
    """One MIM stack centred in the axes: Mo (grounded) / SiOx / top electrode (-V).

    Bias convention matches the device measurement: a negative bias is applied to
    the top electrode and the Mo bottom electrode is grounded, so the field drives
    the positive mobile species (a proton, with oxygen vacancies as the coupled
    species) UP toward the negatively-biased top electrode."""
    ax.set_xlim(0, 4); ax.set_ylim(0, 3.7); ax.axis("off")
    x0, w = 0.7, 2.6
    y_mo, h_mo = 0.25, 0.45
    y_ox, h_ox = 0.70, 1.7
    y_top, h_top = 2.40, 0.45
    # Mo bottom electrode (grounded)
    ax.add_patch(Rectangle((x0, y_mo), w, h_mo, fc=MO, ec=INK, lw=1.0, zorder=2))
    ax.text(x0+w/2, y_mo+h_mo/2, "Mo", ha="center", va="center", fontsize=8, color="white")
    # ground symbol on Mo
    gx = x0 + w + 0.18
    ax.plot([x0+w, gx], [y_mo+h_mo/2, y_mo+h_mo/2], color=INK, lw=1.0)
    for i, hw in enumerate([0.13, 0.08, 0.04]):
        yy = y_mo+h_mo/2 - 0.10 - 0.06*i
        ax.plot([gx-hw, gx+hw], [yy, yy], color=INK, lw=1.0)
    ax.plot([gx, gx], [y_mo+h_mo/2, y_mo+h_mo/2-0.10], color=INK, lw=1.0)
    # SiOx
    ax.add_patch(Rectangle((x0, y_ox), w, h_ox, fc=OXIDE, ec=INK, lw=1.0, zorder=2))
    ax.text(x0+0.08, y_ox+h_ox-0.12, r"SiO$_x$", ha="left", va="top", fontsize=8, color=INK)
    # top electrode + applied -V
    if reduced:
        ax.add_patch(Rectangle((x0, y_top), w, h_top, fc=ITOr, ec=INK, lw=1.0, zorder=2, hatch="////"))
    else:
        ax.add_patch(Rectangle((x0, y_top), w, h_top, fc=top_color, ec=INK, lw=1.0, zorder=2))
    ax.text(x0+w/2, y_top+h_top/2, top_label, ha="center", va="center", fontsize=8, color="white")
    ax.annotate(r"$-V$ applied", xy=(x0+w/2, y_top+h_top), xytext=(x0+w/2, y_top+h_top+0.55),
                ha="center", fontsize=8.0, color=INK,
                arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.3))
    # distributed traps (same pattern in both)
    xs = x0 + 0.18 + (w-0.36)*TRAP_PTS[:, 0]
    ys = y_ox + 0.20 + (h_ox-0.40)*TRAP_PTS[:, 1]
    ax.scatter(xs, ys, s=15, facecolor="white", edgecolor=TRAP, lw=1.0, zorder=4)
    # mobile positive species drifts UP toward the negative top electrode
    for fx in np.linspace(0.28, 0.72, 4):
        x = x0 + fx*w
        ax.annotate("", xy=(x, y_top-0.04), xytext=(x, y_ox+0.35),
                    arrowprops=dict(arrowstyle="-|>", color=PROT, lw=1.3, alpha=0.9))
        ax.text(x+0.07, y_ox+0.62, r"H$^{+}$", color=PROT, fontsize=6.6, va="center", zorder=5)
    return dict(x0=x0, w=w, y_top=y_top, y_ox=y_ox, h_ox=h_ox)

def panel_a(ax):
    ax.text(-0.06, 1.12, "(a)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.text(0.5, 1.02, "Ti/Au: inert contact", transform=ax.transAxes,
            ha="center", fontsize=9.5, fontweight="bold")
    g = _stack(ax, reduced=False, top_color=GOLD, top_label="Au")
    # accumulation layer (+) just under the inert gold
    ax.add_patch(Rectangle((g["x0"], g["y_top"]-0.15), g["w"], 0.15, fc=PROT, ec="none", alpha=0.6, zorder=3))
    for fx in np.linspace(0.15, 0.85, 6):
        ax.text(g["x0"]+fx*g["w"], g["y_top"]-0.075, "+", color="white",
                ha="center", va="center", fontsize=8, fontweight="bold", zorder=5)
    ax.text(2.0, 0.00, r"H$^{+}$ accumulates at inert Au", ha="center", fontsize=8.0, color=PROT)
    ax.text(2.0, -0.30, "$\\rightarrow$ space charge screens the field", ha="center", fontsize=7.8, color=PROT)

def panel_b(ax):
    ax.text(-0.06, 1.12, "(b)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.text(0.5, 1.02, "ITO: reduced by protons", transform=ax.transAxes,
            ha="center", fontsize=9.5, fontweight="bold")
    g = _stack(ax, reduced=True, top_color=ITO, top_label="ITO")
    for fx in np.linspace(0.2, 0.8, 5):
        ax.text(g["x0"]+fx*g["w"], g["y_top"]-0.01, r"$\times$", color=PROT,
                ha="center", va="center", fontsize=9, fontweight="bold", zorder=5)
    ax.text(2.0, 0.00, r"H$^{+}$ reduces ITO, consumed", ha="center", fontsize=8.0, color=GREEN)
    ax.text(2.0, -0.30, "$\\rightarrow$ no space charge accumulates", ha="center", fontsize=7.8, color=GREEN)

def _transient_axes(ax):
    ax.set_xlim(0, 6); ax.set_ylim(0, 1.08)
    ax.set_xlabel("time (s)", fontsize=8.5); ax.set_ylabel(r"$|I|$", fontsize=9)
    ax.tick_params(labelsize=7.5)

def panel_c(ax):
    """Gold transient: rise then slow tau_d decay -> masks the discharge."""
    ax.text(-0.06, 1.12, "(c)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("Au: slow decay dominates", fontsize=9.5, fontweight="bold", pad=4)
    t = np.linspace(0, 6, 400)
    rise = 1 - np.exp(-(t/0.35)**2)
    I = rise*np.exp(-t/9.0); I = I/I.max()
    ax.plot(t, I, color=PROT, lw=2.4)
    _transient_axes(ax)
    # (tau_d arrow and "trap discharge hidden underneath" text removed -- the
    #  caption states the slow proton decay masks the faster trap discharge)

def panel_d(ax):
    """ITO transient: bare dispersive discharge exposed -> tau_leak."""
    ax.text(-0.06, 1.12, "(d)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("ITO: bare discharge exposed", fontsize=9.5, fontweight="bold", pad=4)
    t = np.linspace(0, 6, 400)
    rise = 1 - np.exp(-(t/0.30)**2)
    I = rise*np.exp(-(t/1.3)**0.54); I = I/I.max()
    ax.plot(t, I, color=TRAP, lw=2.4)
    _transient_axes(ax)
    # (tau_leak arrow/annotation removed -- the caption states ITO exposes the
    #  bare discharge tau_leak)

def panel_e(ax):
    """Why dispersive: distribution of rates -> stretched exponential."""
    ax.text(-0.06, 1.12, "(e)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("Why dispersive", fontsize=9.5, fontweight="bold", pad=4)
    t = np.linspace(0, 6, 400)
    for tt in [0.4, 0.8, 1.6, 3.2]:
        ax.plot(t, np.exp(-t/tt), color=NEU, lw=0.9, alpha=0.6)
    ax.plot(t, np.exp(-(t/1.3)**0.54), color=TRAP, lw=2.6, label=r"sum: $\beta\!=\!0.54$")
    ax.plot(t, np.exp(-t/1.3), color=PROT, lw=1.5, ls=(0, (4, 2)), label=r"single: $\beta\!=\!1$")
    ax.set_xlim(0, 6); ax.set_ylim(0, 1.02)
    ax.set_xlabel("time (arb.)", fontsize=8.5); ax.set_ylabel("norm. $I$", fontsize=8.5)
    ax.tick_params(labelsize=7.5)
    ax.legend(fontsize=7.4, loc="upper right", framealpha=0.92)
    # ("grey: individual release rates" floating text removed -- caption explains
    #  the spread of release rates summing to a stretched exponential)

def panel_f(ax):
    """Illustrative compressed-fill and stretched-discharge curve shapes."""
    ax.text(-0.06, 1.12, "(f)", transform=ax.transAxes, fontsize=13, fontweight="bold")
    ax.set_title("Illustrative fitted curve shapes", fontsize=9.5, fontweight="bold", pad=4)
    t = np.linspace(0, 1, 200)
    ax.plot(t, 1-np.exp(-(t/0.18)**2), color=GOLD, lw=2.4, label=r"fill $\beta\!\approx\!2$")
    ax.plot(t, np.exp(-(t/0.30)**0.54), color=TRAP, lw=2.4, label=r"discharge $\beta\!\approx\!0.5$")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
    ax.set_xlabel("time (arb.)", fontsize=8.5); ax.set_ylabel("norm. $I$", fontsize=8.5)
    ax.set_xticks([0, 0.5, 1.0]); ax.tick_params(labelsize=7.5)
    # floating "fill/discharge" labels and the italic subtitle removed; the two
    # curves are distinguished by a compact legend and explained in the caption
    ax.legend(fontsize=7.4, loc="center right", framealpha=0.92)

TRAP_PTS = np.random.RandomState(7).rand(11, 2)
fig, axes = plt.subplots(2, 3, figsize=(11.0, 6.4))
panel_a(axes[0, 0]); panel_b(axes[0, 1]); panel_e(axes[0, 2])
panel_c(axes[1, 0]); panel_d(axes[1, 1]); panel_f(axes[1, 2])
fig.subplots_adjust(left=0.06, right=0.97, top=0.92, bottom=0.12, wspace=0.32, hspace=0.42)

finish(fig, "fig_dispersive.png", "immutable-authored", method_provenance={
    "status": "proposed",
    "established_basis": ["dispersive relaxation", "distributed waiting times"],
    "repository_adaptation": "qualitative Au/ITO transport schematic",
    "claim_limit": "illustrative hypotheses only; no shared microscopic mechanism or trap count is identified",
})


## ITO analysis ownership

Claim-bearing ITO fitting, quality control, measurement-regime separation and
field-law analysis are owned exclusively by `06_nmi_predictive_linkage.ipynb`.
This notebook does not infer workbook identity, refit ITO traces or generate ITO
quantitative figures. The transport schematic above remains explicitly conceptual.


## Intrinsic Trace Timescales

The fitted Au rise/screening ranges, deliberately swept retention values, and
literature biological window are laid on one logarithmic axis. ITO summaries
are intentionally excluded here and are generated only by notebook 06.


In [ ]:
"""Device/biology timescale ladder from the authored generator."""
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

TEAL = "#3aa07a"; INDIGO = "#2f4b8f"; BRICK = "#c0392b"; INK = "#2b2b2b"; GRID = "#c8d0d8"
BIO = (0.3, 10.0)
TAU_R = (1.9, 14.5)
TAU_D = (36.0, 537.0)
TAU_SWEEP = [5, 10, 20]
OVERLAP = (max(BIO[0], TAU_R[0]), min(BIO[1], TAU_R[1]))
Y_BIO, Y_R, Y_D, Y_LEAK, BAR_H = 3, 2, 1, 0, 0.30

fig, ax = plt.subplots(figsize=(7.2, 3.0))
ax.axvspan(OVERLAP[0], OVERLAP[1], color=TEAL, alpha=0.08, zorder=0)
# ax.text(np.sqrt(OVERLAP[0] * OVERLAP[1]), 4.05,
#         "device dynamics meet\nthe biological window", ha="center", va="center",
#         fontsize=7.0, style="italic", color="#5a6b62", linespacing=1.05, zorder=5)
ax.fill_between(BIO, Y_BIO-BAR_H, Y_BIO+BAR_H, facecolor="none", edgecolor=BRICK,
                hatch="////", linewidth=1.1, zorder=3)
# ax.text(BIO[1]*1.35, Y_BIO, "biological eligibility window (~0.3-10 s)",
        # va="center", ha="left", fontsize=8.5, color=BRICK)
ax.fill_between(TAU_R, Y_R-BAR_H, Y_R+BAR_H, color=TEAL, alpha=0.85, zorder=3)
# ax.text(TAU_R[1]*1.35, Y_R, r"device rise  $\tau_r$  (measured, 1.9-14.5 s)",
#         va="center", ha="left", fontsize=8.5, color=INK)
ax.fill_between(TAU_D, Y_D-BAR_H, Y_D+BAR_H, color=TEAL, alpha=0.45, zorder=3)
# ax.text(TAU_D[0]*0.62, Y_D, r"device decay  $\tau_d$  (measured, 36-537 s)",
#         va="center", ha="right", fontsize=8.5, color=INK)
for retention in TAU_SWEEP:
    ax.plot([retention], [Y_LEAK], marker="o", ms=8.5, mfc="white", mec=INDIGO, mew=1.6, zorder=4)
    ax.annotate(f"{retention}", (retention, Y_LEAK), textcoords="offset points",
                xytext=(0, 9), ha="center", fontsize=7, color=INDIGO)
ax.set_xscale("log"); ax.set_xlim(0.1, 1000); ax.set_ylim(-0.7, 4.5)
ax.set_xlabel("time constant / window (s)"); ax.set_yticks([])
for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)
ax.tick_params(axis="x", which="both", length=3)
ax.grid(True, axis="x", which="major", color=GRID, lw=0.6, zorder=0); ax.set_axisbelow(True)
legend_handles = [
    Patch(facecolor=TEAL, alpha=0.85, label="fitted Au dynamics"),
    Line2D([0], [0], marker="o", color="none", mfc="white", mec=INDIGO, mew=1.6, ms=8,
           label="deliberate simulation sweep"),
    Patch(facecolor="none", edgecolor=BRICK, hatch="////", label="biological (literature)"),
]
ax.legend(handles=legend_handles, loc="upper right", bbox_to_anchor=(1.0, 0.99),
          frameon=False, fontsize=7.5, handlelength=1.6, borderaxespad=0.2, labelspacing=0.35)
fig.tight_layout()
finish(fig, "fig_timescales.png", "live-exact", data_paths=[DEVICE_DATA / "kww_final.json"])


## Notebook report


In [ ]:
expected = {
    "fig_measured_transient.png", "fig_trace_shapes.png", "fig_dispersive.png",
    "fig_timescales.png",
}
assert {row["filename"] for row in FIGURE_REPORT} == expected
assert all(row["saved_path"] is None for row in FIGURE_REPORT) if not SAVE_FIGURES else True
FIGURE_REPORT
